# Inference example for single cases

This is not an example for batch processing. For that, see [`example_batchinfer.ipynb`](./example_batchinfer.ipynb).

In [ ]:
import fingernet

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os

images = [
    '../../datasets/NISTSD27/images/B101L9U.bmp',
    '../../datasets/CISL24218/images/A0100003009991600022036_2.bmp',
    '../../datasets/FVC2002DB2A/1_1.bmp',
    '../../datasets/NIST4/F0001_01.bmp',
    '../../datasets/NIST14/F0000001.bmp',
]

def clean_filename(filename: str) -> str:
    # Get the dataset
    return filename.split('/')[3].split('.')[0]

np_images = []
plt.figure(figsize=(10, 2.5))
for image_path in images:
    plt.subplot(1, len(images), images.index(image_path) + 1)
    img = Image.open(image_path).convert('L')
    np_images.append(np.array(img))
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.title(f'{clean_filename(image_path)}', fontsize=8)
plt.tight_layout()

In [ ]:
def plot_fingernet_outputpath(outputpath):
    # *_enh.png
    # *_seg.png
    # *_ori.png
    # *_mnt.png
    plt.figure(figsize=(15, 6))
    suffixes = ['_enh.png', '_seg.png', '_ori.png', '_mnt.png']
    # Get the common name '*_enh.png'
    common_name = [f for f in os.listdir(outputpath) if f.endswith('_enh.png')][0].replace('_enh.png', '')
    for suffix in suffixes:
        plt.subplot(1, len(suffixes), suffixes.index(suffix) + 1)
        img = Image.open(os.path.join(outputpath, common_name + suffix))
        plt.imshow(img, cmap='gray')
        plt.axis('off')
        plt.title(f'{common_name + suffix}', fontsize=8)
    plt.tight_layout()

def plot_fingernet_output(input, output):
    from fingernet.plot import plot_mnt, plot_ori_field
    
    fig, axes = plt.subplots(1, 4, figsize=(15, 5))
    # 1. Enhanced image
    enhanced_image = output['enhanced_image']
    enhanced_image = enhanced_image.squeeze().cpu().numpy()
    axes[0].imshow(enhanced_image, cmap='gray')
    axes[0].set_title('enh', fontsize=8)
    axes[0].axis('off')

    # 2. Segmentation
    segmentation = output['segmentation_mask']
    segmentation = segmentation.squeeze().cpu().numpy()
    axes[1].imshow(segmentation, cmap='gray')
    axes[1].set_title('seg', fontsize=8)
    axes[1].axis('off')

    # 3. Overlayed original image with orientation field
    ori = output['orientation_field']
    axes[2].imshow(input.squeeze().cpu().numpy(), cmap='gray')
    plot_ori_field(axes[2], ori.squeeze().cpu().numpy(), stride=12)
    axes[2].set_title('ori', fontsize=8)
    axes[2].axis('off')

    # 4. Overlayed original image with minutiae
    minutiae = output['minutiae']
    axes[3].imshow(input.squeeze().cpu().numpy(), cmap='gray')
    plot_mnt(axes[3], minutiae[0].cpu().numpy(), r=16)
    axes[3].set_title('mnt', fontsize=8)
    axes[3].axis('off')

    plt.suptitle('Fingernet output', fontsize=12)
    plt.tight_layout()

In [ ]:
import logging
import torch
from torchvision import transforms

IDX = '*'

logger = fingernet.get_fingernet_logger('fingernet.model')
logger.setLevel(logging.DEBUG)

fnet = fingernet.get_fingernet()
dev = next(fnet.parameters()).device

if IDX == '*':
    # Plot all images
    for i in range(len(images)):
        input_tensor = torch.tensor(np_images[i]).unsqueeze(0).unsqueeze(0) / 255.0
        input_tensor = input_tensor.to(dev)

        output = fnet(input_tensor)

        plot_fingernet_outputpath(f'../../datasets/output/20251014-182614/{i}')
        plot_fingernet_output(input_tensor, output)
else:
    input_tensor = torch.tensor(np_images[IDX]).unsqueeze(0).unsqueeze(0) / 255.0
    input_tensor = input_tensor.to(dev)
    output = fnet(input_tensor)
    plot_fingernet_outputpath(f'../../datasets/output/20251014-182614/{IDX}')
    plot_fingernet_output(input_tensor, output)